In [ ]:
# TRABALHO FINAL: Análise Estatística com TCL, IC, ANOVA e Distribuições Assimétricas
# ====================================================================================
# Objetivo: Aplicar análises da disciplina em projeto de pesquisa
# Foco: Teorema Central do Limite, Intervalo de Confiança, ANOVA e População Assimétrica (Exponencial)

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import optimize
import warnings
warnings.filterwarnings('ignore')

# Configurar estilos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Bibliotecas carregadas com sucesso!")
print("\n📊 ANÁLISE ESTATÍSTICA - TRABALHO FINAL")
print("=" * 60)

# 📊 Análise Estatística: TCL, IC, ANOVA e Distribuições Assimétricas

## 🎯 Objetivo
Aplicar análises da disciplina (Planejamento e Análise Estatística de Experimentos) em projeto de pesquisa.

## 📋 Tópicos Abordados
- **TCL (Teorema Central do Limite)**: Comportamento da distribuição das médias amostrais
- **IC (Intervalo de Confiança)**: Estimação intervalar de parâmetros populacionais
- **ANOVA (Análise de Variância)**: Comparação entre múltiplos grupos
- **Distribuições Assimétricas**: População com distribuição Exponencial

## 📈 Estratégia
1. Gerar dados de uma população exponencial (assimétrica)
2. Aplicar TCL e verificar convergência para distribuição normal
3. Calcular intervalos de confiança (95%)
4. Realizar ANOVA para comparar grupos
5. Visualizar todos os resultados

In [ ]:
# 1. GERAÇÃO DE DADOS - POPULAÇÃO EXPONENCIAL (ASSIMÉTRICA)
# ==============================================================

# Parâmetros da população exponencial
lambda_param = 2.0  # Taxa de decaimento
tamanho_populacao = 10000
np.random.seed(42)

# Gerar população exponencial
populacao = np.random.exponential(scale=1/lambda_param, size=tamanho_populacao)

# Estatísticas da população
mu_pop = np.mean(populacao)
sigma_pop = np.std(populacao, ddof=0)
assimetria_pop = stats.skew(populacao)
curtose_pop = stats.kurtosis(populacao)

print("\n📌 POPULAÇÃO EXPONENCIAL")
print("-" * 60)
print(f"Tamanho da população: {tamanho_populacao:,}")
print(f"Média populacional (μ): {mu_pop:.4f}")
print(f"Desvio padrão populacional (σ): {sigma_pop:.4f}")
print(f"Assimetria: {assimetria_pop:.4f} (distribuição fortemente assimétrica à direita)")
print(f"Curtose (excesso): {curtose_pop:.4f}")
print(f"Valores: mín={populacao.min():.4f}, máx={populacao.max():.4f}")

# Visualizar a população
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(populacao, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(mu_pop, color='red', linestyle='--', linewidth=2, label=f'Média = {mu_pop:.2f}')
axes[0].set_xlabel('Valores')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição da População Exponencial')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Q-Q Plot
stats.probplot(populacao, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot: População vs Distribuição Normal')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Dados populacionais gerados com sucesso!")

In [ ]:
# 2. TEOREMA CENTRAL DO LIMITE (TCL)
# ====================================

# Diferentes tamanhos de amostra
tamanhos_amostra = [5, 10, 30, 100, 500]
num_amostras = 10000

print("\n📌 TEOREMA CENTRAL DO LIMITE")
print("-" * 60)
print(f"Número de amostras para cada tamanho: {num_amostras:,}")

# Armazenar médias amostrais para cada tamanho
resultados_tcl = {}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, n in enumerate(tamanhos_amostra):
    # Gerar amostras e calcular suas médias
    medias = np.array([np.mean(np.random.choice(populacao, size=n)) for _ in range(num_amostras)])
    resultados_tcl[n] = medias
    
    # Estatísticas
    media_das_medias = np.mean(medias)
    dp_das_medias = np.std(medias, ddof=1)
    teorico_dp_das_medias = sigma_pop / np.sqrt(n)
    assimetria_medias = stats.skew(medias)
    
    print(f"\nTamanho de amostra: n = {n}")
    print(f"  Média das médias: {media_das_medias:.4f} (esperado: {mu_pop:.4f})")
    print(f"  DP das médias: {dp_das_medias:.4f} (teórico: {teorico_dp_das_medias:.4f})")
    print(f"  Assimetria: {assimetria_medias:.4f} (diminui conforme n aumenta)")
    
    # Visualização
    ax = axes[idx]
    ax.hist(medias, bins=50, color='skyblue', alpha=0.7, edgecolor='black', density=True)
    
    # Sobrepor distribuição normal teórica
    x = np.linspace(medias.min(), medias.max(), 100)
    y_normal = stats.norm.pdf(x, mu_pop, teorico_dp_das_medias)
    ax.plot(x, y_normal, 'r-', linewidth=2, label='Distribuição Normal Teórica')
    
    ax.axvline(media_das_medias, color='green', linestyle='--', linewidth=2, label='Média observada')
    ax.set_xlabel('Média da Amostra')
    ax.set_ylabel('Densidade')
    ax.set_title(f'TCL: n = {n}')
    ax.legend()
    ax.grid(alpha=0.3)

# Remover última subplot vazia
axes[-1].remove()

plt.tight_layout()
plt.show()

print("\n✅ TCL verificado: As médias amostrais convergem para distribuição normal conforme n aumenta!")

In [ ]:
# 3. INTERVALO DE CONFIANÇA (IC)
# ===============================

print("\n📌 INTERVALO DE CONFIANÇA")
print("-" * 60)

# Parâmetros do IC
nivel_confianca = 0.95
alpha = 1 - nivel_confianca
tamanho_amostra_ic = 50
num_amostras_ic = 1000

# Gerar amostras
amostras_ic = [np.random.choice(populacao, size=tamanho_amostra_ic) for _ in range(num_amostras_ic)]

# Calcular ICs para cada amostra
ics = []
contem_media = 0

for amostra in amostras_ic:
    # Média e erro padrão da amostra
    x_bar = np.mean(amostra)
    s = np.std(amostra, ddof=1)
    se = s / np.sqrt(tamanho_amostra_ic)
    
    # Valor crítico t (distribuição t-student)
    t_critico = stats.t.ppf(1 - alpha/2, df=tamanho_amostra_ic - 1)
    
    # IC
    margem_erro = t_critico * se
    ic_inf = x_bar - margem_erro
    ic_sup = x_bar + margem_erro
    
    ics.append((ic_inf, ic_sup))
    
    # Verificar se contém a média populacional
    if ic_inf <= mu_pop <= ic_sup:
        contem_media += 1

print(f"Tamanho de amostra: n = {tamanho_amostra_ic}")
print(f"Número de amostras: {num_amostras_ic}")
print(f"Nível de confiança: {nivel_confianca*100:.0f}%")
print(f"\nICs que contêm a média populacional: {contem_media}/{num_amostras_ic} ({contem_media/num_amostras_ic*100:.1f}%)")
print(f"Esperado: ~{nivel_confianca*100:.0f}%")

# Visualizar alguns ICs
num_vis = 100
fig, ax = plt.subplots(figsize=(12, 8))

for i in range(num_vis):
    ic_inf, ic_sup = ics[i]
    cor = 'green' if ic_inf <= mu_pop <= ic_sup else 'red'
    ax.plot([ic_inf, ic_sup], [i, i], color=cor, alpha=0.6, linewidth=1)
    ax.plot(np.mean([ic_inf, ic_sup]), i, 'o', color=cor, markersize=3)

ax.axvline(mu_pop, color='blue', linestyle='--', linewidth=2, label=f'Média populacional = {mu_pop:.2f}')
ax.set_xlabel('Valor')
ax.set_ylabel(f'Amostra (primeiras {num_vis})')
ax.set_title(f'Intervalos de Confiança {nivel_confianca*100:.0f}% para μ (n={tamanho_amostra_ic})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ ICs calculados: Verde = contém μ, Vermelho = não contém μ")

In [ ]:
# 4. ANOVA - ANÁLISE DE VARIÂNCIA
# =================================

print("\n📌 ANOVA - ANÁLISE DE VARIÂNCIA")
print("-" * 60)

# Criar 3 grupos com diferentes níveis de um fator
num_grupos = 3
tamanho_grupo = 50

# Grupos com transformações da distribuição exponencial
grupo1 = np.random.exponential(scale=1/2.0, size=tamanho_grupo)  # λ=2.0
grupo2 = np.random.exponential(scale=1/1.5, size=tamanho_grupo) + 0.5  # λ=1.5, deslocado
grupo3 = np.random.exponential(scale=1/1.0, size=tamanho_grupo) + 1.0  # λ=1.0, deslocado

grupos = [grupo1, grupo2, grupo3]
labels_grupos = ['Grupo 1 (λ=2.0)', 'Grupo 2 (λ=1.5, +0.5)', 'Grupo 3 (λ=1.0, +1.0)']

# Estatísticas descritivas
print("\nEstatísticas dos grupos:")
for i, (grupo, label) in enumerate(zip(grupos, labels_grupos)):
    print(f"\n{label}")
    print(f"  Tamanho: n = {len(grupo)}")
    print(f"  Média: {np.mean(grupo):.4f}")
    print(f"  Desvio padrão: {np.std(grupo, ddof=1):.4f}")
    print(f"  Mín: {np.min(grupo):.4f}, Máx: {np.max(grupo):.4f}")

# Realizar ANOVA (One-way ANOVA)
f_stat, p_value = stats.f_oneway(grupo1, grupo2, grupo3)

print(f"\n{'RESULTADO DA ANOVA':^60}")
print(f"Estatística F: {f_stat:.4f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.05:
    print("✅ Resultado: SIGNIFICATIVO (rejeita-se H₀)")
    print("   Conclusão: Há diferenças estatísticas significativas entre os grupos")
else:
    print("❌ Resultado: NÃO SIGNIFICATIVO (falha em rejeitar H₀)")
    print("   Conclusão: Não há diferenças estatísticas significativas entre os grupos")

# Visualizações
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Boxplot
ax1 = axes[0]
bp = ax1.boxplot(grupos, labels=labels_grupos, patch_artist=True)
for patch, color in zip(bp['boxes'], ['lightblue', 'lightgreen', 'lightcoral']):
    patch.set_facecolor(color)
ax1.set_ylabel('Valores')
ax1.set_title('Boxplot dos Grupos (ANOVA)')
ax1.grid(alpha=0.3, axis='y')
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=15, ha='right')

# Histogramas sobrepostos
ax2 = axes[1]
cores = ['blue', 'green', 'red']
for grupo, label, cor in zip(grupos, labels_grupos, cores):
    ax2.hist(grupo, bins=20, alpha=0.5, label=label, color=cor, edgecolor='black')
ax2.set_xlabel('Valores')
ax2.set_ylabel('Frequência')
ax2.set_title('Distribuição dos Grupos')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ ANOVA realizada com sucesso!")

In [ ]:
# 5. RESUMO INTEGRADO: TCL + IC + ANOVA + DISTRIBUIÇÃO ASSIMÉTRICA
# ===================================================================

print("\n📌 RESUMO INTEGRADO DA ANÁLISE")
print("=" * 60)

print("\n1️⃣ POPULAÇÃO EXPONENCIAL (ASSIMÉTRICA)")
print(f"   - Distribuição: Exponencial com λ = {lambda_param}")
print(f"   - Média: {mu_pop:.4f}")
print(f"   - Desvio padrão: {sigma_pop:.4f}")
print(f"   - Assimetria: {assimetria_pop:.4f} (forte assimetria à direita)")
print(f"   - Característica: População NÃO é normal")

print("\n2️⃣ TEOREMA CENTRAL DO LIMITE")
print(f"   - Verificado: Médias amostrais convergem para normal")
print(f"   - Conforme n aumenta:")
print(f"     • Distribuição das médias → Normal")
print(f"     • Assimetria das médias → 0")
print(f"   - TCL funciona mesmo com população assimétrica!")

print("\n3️⃣ INTERVALO DE CONFIANÇA (95%)")
print(f"   - Tamanho de amostra: n = {tamanho_amostra_ic}")
print(f"   - Cobertura empírica: {contem_media/num_amostras_ic*100:.1f}%")
print(f"   - Cobertura teórica: {nivel_confianca*100:.1f}%")
print(f"   - Conclusão: IC funciona bem mesmo com dados não-normais")

print("\n4️⃣ ANOVA (Análise de Variância)")
print(f"   - Grupos comparados: {num_grupos}")
print(f"   - Tamanho de cada grupo: {tamanho_grupo}")
print(f"   - Estatística F: {f_stat:.4f}")
print(f"   - p-value: {p_value:.6f}")
if p_value < 0.05:
    print(f"   - Resultado: SIGNIFICATIVO (α = 0.05)")
    print(f"   - Interpretação: Grupos têm médias diferentes")
else:
    print(f"   - Resultado: NÃO SIGNIFICATIVO (α = 0.05)")
    print(f"   - Interpretação: Grupos têm médias semelhantes")

print("\n5️⃣ ROBUSTEZ DOS MÉTODOS")
print(f"   - TCL: ✅ Robusto para distribuições não-normais")
print(f"   - IC (t-student): ✅ Robusto para distribuições não-normais")
print(f"   - ANOVA: ✅ Razoavelmente robusto para distribuições não-normais")
print(f"   - Recomendação: n ≥ 30 para garantir validade dos testes")

# Visualização final integrada
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

# 1. População original
ax1 = fig.add_subplot(gs[0, :2])
ax1.hist(populacao, bins=50, color='steelblue', alpha=0.7, edgecolor='black', density=True)
ax1.axvline(mu_pop, color='red', linestyle='--', linewidth=2, label=f'μ = {mu_pop:.2f}')
ax1.set_xlabel('Valores')
ax1.set_ylabel('Densidade')
ax1.set_title(f'1. População Exponencial (Assimétrica) - Assimetria = {assimetria_pop:.2f}')
ax1.legend()
ax1.grid(alpha=0.3)

# 2. TCL: distribuição das médias (n=100)
ax2 = fig.add_subplot(gs[0, 2])
medias_100 = resultados_tcl[100]
ax2.hist(medias_100, bins=40, color='skyblue', alpha=0.7, edgecolor='black', density=True)
x_norm = np.linspace(medias_100.min(), medias_100.max(), 100)
y_norm = stats.norm.pdf(x_norm, mu_pop, sigma_pop/np.sqrt(100))
ax2.plot(x_norm, y_norm, 'r-', linewidth=2, label='Normal teórica')
ax2.set_xlabel('Valores')
ax2.set_ylabel('Densidade')
ax2.set_title('2. TCL: Médias (n=100)')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# 3. Convergência do TCL
ax3 = fig.add_subplot(gs[1, 0])
assimetrias = []
for n in tamanhos_amostra:
    assimetrias.append(stats.skew(resultados_tcl[n]))
ax3.plot(tamanhos_amostra, assimetrias, 'o-', linewidth=2, markersize=8, color='green')
ax3.axhline(0, color='red', linestyle='--', linewidth=1)
ax3.set_xlabel('Tamanho da amostra (n)')
ax3.set_ylabel('Assimetria')
ax3.set_title('3. Convergência TCL: Assimetria → 0')
ax3.set_xscale('log')
ax3.grid(alpha=0.3)

# 4. IC - Cobertura
ax4 = fig.add_subplot(gs[1, 1])
num_ic_vis = 100
for i in range(num_ic_vis):
    ic_inf, ic_sup = ics[i]
    cor = 'green' if ic_inf <= mu_pop <= ic_sup else 'red'
    ax4.plot([ic_inf, ic_sup], [i, i], color=cor, alpha=0.4, linewidth=0.8)
ax4.axvline(mu_pop, color='blue', linestyle='--', linewidth=2, label=f'μ = {mu_pop:.2f}')
ax4.set_xlabel('Valor')
ax4.set_ylabel('Amostra')
ax4.set_title(f'4. IC 95%: Cobertura = {contem_media/num_amostras_ic*100:.1f}%')
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3, axis='x')

# 5. ANOVA - Boxplot
ax5 = fig.add_subplot(gs[1, 2])
bp = ax5.boxplot(grupos, labels=[f'G{i+1}' for i in range(num_grupos)], patch_artist=True)
for patch, color in zip(bp['boxes'], ['lightblue', 'lightgreen', 'lightcoral']):
    patch.set_facecolor(color)
ax5.set_ylabel('Valores')
ax5.set_title(f'5. ANOVA: F = {f_stat:.2f}, p = {p_value:.4f}')
ax5.grid(alpha=0.3, axis='y')

# 6. Q-Q Plot da população
ax6 = fig.add_subplot(gs[2, 0])
stats.probplot(populacao, dist="norm", plot=ax6)
ax6.set_title('6. Q-Q Plot: População vs Normal')
ax6.grid(alpha=0.3)

# 7. Q-Q Plot das médias amostrais (n=100)
ax7 = fig.add_subplot(gs[2, 1])
stats.probplot(resultados_tcl[100], dist="norm", plot=ax7)
ax7.set_title('7. Q-Q Plot: Médias (n=100) vs Normal')
ax7.grid(alpha=0.3)

# 8. Resumo textual
ax8 = fig.add_subplot(gs[2, 2])
ax8.axis('off')
resumo_texto = f"""
CONCLUSÕES:

✅ TCL: Funciona mesmo 
   com dados não-normais

✅ IC: Cobertura = 
   {contem_media/num_amostras_ic*100:.1f}%
   
✅ ANOVA: 
   F={f_stat:.2f}, p={p_value:.4f}
   
📊 Métodos robustos a
   distribuições assimétricas
"""
ax8.text(0.1, 0.5, resumo_texto, fontsize=10, family='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', 
         facecolor='lightyellow', alpha=0.8))

plt.suptitle('ANÁLISE INTEGRADA: TCL + IC + ANOVA + DISTRIBUIÇÃO ASSIMÉTRICA', 
             fontsize=14, fontweight='bold', y=0.995)
plt.show()

print("\n" + "="*60)
print("✅ ANÁLISE COMPLETA FINALIZADA COM SUCESSO!")
print("="*60)